# Tool Calling Fundamentals
### Practice Notebook

## 1. What is a tool, mechanically?

A tool is: a **name**, a **description** (tells the model *when* to use it),
and an **input schema** (tells the model *what arguments* it needs to
provide). The model never runs the function itself -- it only ever outputs a
*request* to call it; your code executes the real function and reports the
result back.

Let's define a real Python function, then wrap it with a tool schema in the
same shape the Anthropic API expects.


In [1]:
def get_weather(city: str, unit: str = "celsius") -> dict:
    """The REAL function that will actually run. In a production project
    this might call a weather API; here it returns fake data so the
    notebook works offline.
    """
    fake_temperatures = {"Bengaluru": 24, "Mumbai": 31, "Delhi": 19}
    temp_c = fake_temperatures.get(city, 25)
    if unit == "fahrenheit":
        return {"city": city, "temperature": temp_c * 9 / 5 + 32, "unit": "fahrenheit"}
    return {"city": city, "temperature": temp_c, "unit": "celsius"}

# The TOOL SCHEMA describing get_weather to an LLM
get_weather_tool = {
    "name": "get_weather",
    "description": (
        "Get the current weather for a specific city. Use this whenever the "
        "user asks about current temperature or weather conditions in a "
        "named city. Do not use this for weather forecasts (future dates)."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city name, e.g. 'Bengaluru' or 'Mumbai'."
            },
            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Temperature unit. Defaults to celsius if not specified."
            }
        },
        "required": ["city"]
    }
}

import json
print(json.dumps(get_weather_tool, indent=2))


{
  "name": "get_weather",
  "description": "Get the current weather for a specific city. Use this whenever the user asks about current temperature or weather conditions in a named city. Do not use this for weather forecasts (future dates).",
  "input_schema": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string",
        "description": "The city name, e.g. 'Bengaluru' or 'Mumbai'."
      },
      "unit": {
        "type": "string",
        "enum": [
          "celsius",
          "fahrenheit"
        ],
        "description": "Temperature unit. Defaults to celsius if not specified."
      }
    },
    "required": [
      "city"
    ]
  }
}


**Exercise 2.1:** Identify the three things above that make this a *good*
tool definition: what specifically in the `description` tells the model when
*not* to use this tool? What would happen (in terms of model behavior) if
that sentence were deleted?


## 2. Simulating the model's tool-selection decision

We don't have a live model call here, so let's build a tiny **stand-in
router** that mimics what an LLM does when deciding which tool (if any) to
call, based on keyword matching. This is a crude approximation -- a real
LLM uses the full semantic content of the tool description and the user's
message, not keyword rules -- but it makes the *decision step* itself
visible and inspectable.


In [2]:
available_tools = [
    {
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "keywords": ["weather", "temperature", "hot", "cold", "rain"],
    },
    {
        "name": "convert_currency",
        "description": "Convert an amount from one currency to another.",
        "keywords": ["convert", "currency", "usd", "inr", "exchange rate"],
    },
    {
        "name": "search_flights",
        "description": "Search for flights between two cities.",
        "keywords": ["flight", "fly", "airline", "book a ticket"],
    },
]

def stub_tool_router(user_message: str) -> str | None:
    """Stand-in for: 'the model decides which tool (if any) to call'.
    # TODO: replace this whole function with a real LLM call that receives
    # `available_tools` (in real schema form) and the user_message, and
    # inspect response.content for a tool_use block instead.
    """
    msg_lower = user_message.lower()
    for tool in available_tools:
        if any(kw in msg_lower for kw in tool["keywords"]):
            return tool["name"]
    return None

test_messages = [
    "What's the weather like in Delhi right now?",
    "How much is 100 US dollars in rupees?",
    "Find me a flight from Bengaluru to Chennai.",
    "Tell me a joke.",
]

for msg in test_messages:
    chosen = stub_tool_router(msg)
    print(f"'{msg}'  ->  tool: {chosen}")


'What's the weather like in Delhi right now?'  ->  tool: get_weather
'How much is 100 US dollars in rupees?'  ->  tool: None
'Find me a flight from Bengaluru to Chennai.'  ->  tool: search_flights
'Tell me a joke.'  ->  tool: None


**Exercise 2.2:** The last message ("Tell me a joke.") correctly returns
`None` -- no tool fits. Add a new message that's *ambiguous* between two
tools (e.g., something that mentions both weather and flights) and see which
one `stub_tool_router` picks. Is that a good decision? What does this reveal
about a real limitation of keyword-based routing that a real LLM (reasoning
over full context, not just keyword presence) would generally handle
better?


## 3. The full round-trip: call -> execute -> report result -> final answer

In [ ]:
def full_tool_use_round_trip(user_message: str):
    print(f"User: {user_message}")

    # Step 1: model decides a tool call is needed
    tool_name = stub_tool_router(user_message)
    if tool_name is None:
        print("Assistant: (no tool needed) (stub) responding directly.")
        return

    # Step 2: (simplified) extract an argument -- a real model would parse
    # this out of the user's message using its own reasoning; we hard-code
    # a simple extraction for this toy example.
    import re
    city_match = re.search(r"in (\w+)", user_message)
    city = city_match.group(1) if city_match else "Bengaluru"

    print(f"Assistant requests tool call: {tool_name}(city='{city}')")

    # Step 3: YOUR CODE actually executes the function (the model never does this)
    if tool_name == "get_weather":
        result = get_weather(city)
    else:
        result = {"error": f"no implementation wired up for {tool_name} in this demo"}

    print(f"Tool result returned to model: {result}")

    # Step 4: model uses the tool result to produce a final answer
    # TODO: replace with a real LLM call that receives `result` as a tool_result
    print(f"Assistant (final): The weather in {result.get('city', city)} "
          f"is {result.get('temperature', '?')}°{result.get('unit', 'celsius')[0].upper()}.")

full_tool_use_round_trip("What's the weather like in Mumbai right now?")


**Exercise 2.3 (mini deliverable):** Implement a second real tool function
`convert_currency(amount, from_currency, to_currency)` (use a small hard-coded
exchange-rate dictionary, no real API needed) plus its tool schema (name,
description, input_schema) in the same style as `get_weather_tool`. Then
extend `full_tool_use_round_trip` so it actually calls your new tool for
currency-related messages, not just weather ones.


## 4. (Optional) Wiring in a real LLM call

If you have an `ANTHROPIC_API_KEY` available, here's how the stub functions
above map onto the real Anthropic tool-use API. This cell is optional and
will not run without a valid key -- read through it even if you can't
execute it, since it shows exactly what the stubs above were standing in
for.


In [ ]:
# client = InferenceClient(api_key=HF_TOKEN)
# response = client.chat.completions.create(
#     model="Qwen/Qwen2.5-7B-Instruct",
#     max_tokens=500,
#     tools=[get_weather_tool],
#     messages=[{"role": "user", "content": "What's the weather like in Mumbai right now?"}]
# )
#
# # The model's response will contain a tool_use block if it decided to call the tool:
# for block in response.choices[0].message.content:
#     if block.type == "tool_use":
#         print("Model wants to call:", block.name, "with input:", block.input)
#         tool_result = get_weather(**block.input)   # YOUR code actually runs it
#         # ... then send tool_result back in a new message with role='user',
#         # content=[{"type": "tool_result", "tool_use_id": block.id, "content": str(tool_result)}]

print("This cell is illustrative -- uncomment and add your API key to run it for real.")
